Coding Quiz: Week 10 (Due in Week 11)

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('/Users/gkumargaur/workspace/datascience/boston_university/semester3/ms_datascience/data/homework_10.1.csv')

print('Total number of rows in the dataset:', len(df))
df.head()

Total number of rows in the dataset: 360


,city,time,X,y
0,0,0,0.144044,7.552716
1,0,1,1.454274,10.077829
2,0,2,0.761038,12.372731
3,0,3,0.121675,11.489263
4,0,4,0.443863,13.104833


Question 1

Use the data in homework_10.1.csv and find the fixed effect (the constant term in the regression) for each time (0 through 11). Which of these describes the pattern: 

Option A
- The effects gradually increase from month 0 to 11. 

Option B
- The effects vary randomly.

Option C
- The effects increase from month 0 to about 5 or 6 but then eventually decrease again.

Option D
- The effects decrease from month 0 to about 5 or 6 but then eventually increase again.

In [3]:
m_time = smf.ols('y ~ X + C(time) - 1', data=df).fit()
print(m_time.params)

C(time)[0]     2.879246
C(time)[1]     4.676858
C(time)[2]     6.148259
C(time)[3]     7.162997
C(time)[4]     8.117890
C(time)[5]     9.016350
C(time)[6]     8.626309
C(time)[7]     8.515667
C(time)[8]     8.407278
C(time)[9]     7.467505
C(time)[10]    5.991741
C(time)[11]    4.623149
X              1.238139
dtype: float64


The effects rise smoothly to a peak at month 5–6, then fall again, like a seasonal hump. Adding city fixed effects as well (y ~ X + C(city) + C(time)) gives the same shape: +1.7 → +6.1 at month 5 → +1.7.

---

Question 2

What about the fixed effect for each city (0 through 9)? 

Option A
- The effects gradually increase from city 0 to 9.

Option B
- The effects vary randomly.

Option C
- The effects increase from city 0 to about 4 or 5 but then eventually decrease again.

Option D
- The effects decrease from city 0 to about 4 or 5 but then eventually increase again. 

In [4]:
m_city = smf.ols('y ~ X + C(city) - 1', data=df).fit()
print(m_city.params)

C(city)[0]    11.029943
C(city)[1]     5.227050
C(city)[2]     7.921836
C(city)[3]    12.926031
C(city)[4]    11.296032
C(city)[5]    -0.057783
C(city)[6]     7.704997
C(city)[7]     3.257647
C(city)[8]     3.384002
C(city)[9]     5.450428
X              1.532467
dtype: float64


The values jump around with no trend and no hump, so they vary randomly. That makes sense: city numbers are just labels, unlike months, which are in order.

---

Question 3


For the following data, model np.exp(Y) as a function of X and Z. 



num = 10000 

 

X = np.clip(np.random.normal(3, 1, (num,)), 0.01, 100)

Z = np.clip(np.random.normal(3, 1, (num,)), 0.01, 100)

Y = np.log(X + Z) + np.random.normal(0, 1, (num,)) 



With enough data, the coefficients of X and Z are closest to: 

Option A
- Both 1 

Option B
- Both 1.3 

Option C
- Both 1.6

Option D
- Both 2.5

In [9]:
num = 10000
X = np.clip(np.random.normal(3, 1, num), 0.01, 100)
Z = np.clip(np.random.normal(3, 1, num), 0.01, 100)
Y = np.log(X + Z) + np.random.normal(0, 1, num)

d = pd.DataFrame({'X': X, 'Z': Z, 'eY': np.exp(Y)})
print(smf.ols('eY ~ X + Z', data=d).fit().params)

Intercept    0.125026
X            1.645943
Z            1.683305
dtype: float64


**Why it isn't 1:** With noise $\varepsilon \sim N(0, 1)$:

$$
e^Y = (X+Z)\,e^{\varepsilon}, \qquad E[e^Y \mid X,Z] = (X+Z)\,E[e^{\varepsilon}] = (X+Z)\,e^{1/2}
$$

Since $e^{0.5} \approx 1.65$, both true coefficients are about $1.65$. The noise multiplies the result instead of adding to it, and the average of $e^{\varepsilon}$ is $e^{\sigma^2/2}$, not $1$. The skewed noise makes single runs bounce around $1.65$.

---

Question 4


Suppose we were to use the data below to find the standard error of X's coefficient (the coefficient that should be 1.5) in two ways: 

1)  By asking Python to find the standard error. 

2)  By simulating what happens if we generate the data 100 times, estimating X's coefficient and finding the standard deviation of the 100 estimates, using this model: 



     num = 10000 

     Z = np.random.normal(0, 1, (num,)) 

     X = Z + np.random.normal(0, 1, (num,)) 

     Y = 1.5 * X + 2.3 * Z + np.random.normal(0, X**2, (num,)) 

           

 Then:

Option A
- 1) is significantly bigger than 2), but neither is zero. 

Option B
- 2) is significantly bigger than 1), but neither is zero.

Option C
- They are about the same. 

Option D
- One or both of them are zero.

In [10]:
num = 10000
est, ses = [], []
for _ in range(100):
    Z = np.random.normal(0, 1, num)
    X = Z + np.random.normal(0, 1, num)
    Y = 1.5*X + 2.3*Z + np.random.normal(0, X**2, num)
    r = smf.ols('Y ~ X + Z', data=pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})).fit()
    est.append(r.params['X']); ses.append(r.bse['X'])

print('Python SE (avg):', np.mean(ses))
print('Simulated SD   :', np.std(est))

Python SE (avg): 0.034621399401752344
Simulated SD   : 0.05330094197774835


**Why:** The noise has standard deviation $X^2$, so it grows as $|X|$ grows. That's heteroskedasticity. Python's default SE assumes the noise is the same everywhere, and here the biggest errors fall on the points with the most pull on the slope. So the default understates the real uncertainty. The simulated spread (2) is about $1.8\times$ bigger, and neither value is zero.